# TP : Partie Cyber + Matching Elasticsearch

Ce notebook regroupe **deux parties indépendantes** :

1. **Partie Cyber** — Tests de connectivité réseau (FQDN, DNS, TCP, HTTPS) sur des sites publics (badssl.com, space-match.net).
2. **Partie Matching** — Découverte du matching par vecteurs avec Elasticsearch (index, bulk, script_score) et bonus sur la pondération.


## 0. Préparation (commune)

- Ce dépôt cloné en local dans `TP_matching`.
- Un interpréteur Python pour la partie Matching ; pour la **partie Cyber**, les cellules sont en **PowerShell** (choisir le noyau PowerShell pour exécuter les cellules de la section 1, ou les copier dans un terminal PowerShell).

**Pour la partie Matching :** Elasticsearch est exécuté **en local** (sans Docker : script d’installation, ou avec Docker), pas de ressource distante (voir section 2).

In [ ]:
import sys
print("Python:", sys.version)

## 1. Partie Cyber : tests de connectivité (indépendante d’Elasticsearch)


### 1.1. Le FQDN (Fully Qualified Domain Name)

Pour joindre un serveur sur Internet, on utilise en général un **nom de domaine complet** (FQDN) (ex **badssl.com**) plutôt que son adresse IP. On va voir comment tester la résolution DNS, la connexion TCP et HTTPS.

### 1.2. Adresse IP et résolution DNS

Les machines sur le réseau sont identifiées par une **adresse IP**. Le **DNS** (Domain Name System) fait le lien entre un nom de domaine (FQDN) et une ou plusieurs adresses IP : c’est la **résolution DNS**. La commande **nslookup** permet d’obtenir l’IP associée à un nom.

Copiez/collez la commande ci-dessous dans un terminal **PowerShell** (elle n’est pas exécutée dans le notebook).


```powershell
nslookup badssl.com
```

### 1.3. Connexion TCP et port 443

Une fois l’IP connue, il faut établir une **connexion TCP** vers le bon **port**. Pour le trafic HTTPS, le port standard est **443**. **Test-NetConnection** permet de vérifier si la machine peut ouvrir une connexion TCP vers ce port.

En **sécurité réseau**, une politique courante est de **bloquer** certains ports. Si on lance **Test-NetConnection** sans préciser le port, c’est le **ping (ICMP)** qui est testé ; un pare-feu peut autoriser le port 443 mais bloquer le ping.

```powershell
Test-NetConnection badssl.com -Port 443
```

Sans port = test ICMP (ping). Un pare-feu peut bloquer le ping.

```powershell
Test-NetConnection badssl.com
```

### 1.4. HTTPS et Invoke-WebRequest

Le protocole **HTTPS** assure le chiffrement et l’authentification du serveur (certificat). Pour tester qu’une URL HTTPS répond correctement en PowerShell : **Invoke-WebRequest** avec **-UseBasicParsing** (pour éviter l’exécution de scripts côté client).

```powershell
Invoke-WebRequest -Uri "https://badssl.com" -UseBasicParsing
```

### 1.5. Exercice à compléter : space-match.net

1. **Trouver l’adresse IP** 
2. **Tester la connexion TCP** vers ce serveur sur le port **443**
3. **Tester la couche HTTPS**


## 2. Partie Matching : Elasticsearch en local et index de 4 produits

On travaille avec un **Elasticsearch lancé en local** (sans Docker ou avec Docker), sans ressource distante. L’index `products` contient **4 produits** avec :
- un champ vecteur principal `product_vector` (dims 4),
- un champ `weights_vector` (dims 4).

**Prérequis :** avoir exécuté une fois `install/install.ps1` (Windows) ou `install/install.sh` (Linux/macOS) : Elasticsearch est alors installé en local, démarré en arrière-plan et l’index `products` (4 produits) est déjà créé.
L'index et les 4 produits sont déjà créés par le script d'installation ; inutile de refaire un PUT ou un bulk.

Le **mapping** de l'index définit les champs suivants : 
- **`name`** (texte, type `text`) 
- **`category`** (catégorie fixe, type `keyword`)
- **`product_vector`** (vecteur dense de dimension 4, utilisé pour le matching)
-  **`weights_vector`** (vecteur dense de dimension 4, pour la pondération). 
```json
{
  "mappings": {
    "properties": {
      "name": {
        "type": "text"
      },
      "category": {
        "type": "keyword"
      },
      "product_vector": {
        "type": "dense_vector",
        "dims": 4
      },
      "weights_vector": {
        "type": "dense_vector",
        "dims": 4
      }
    }
  }
}


### 2.1 Requête KNN simple

Elasticsearch permet une recherche **k plus proches voisins** (KNN) sur un champ `dense_vector` : on fournit un vecteur de requête et le paramètre `k` ; les documents les plus proches (par similarité cosinus ou distance L2) sont retournés. Exemple ci-dessous avec le champ `product_vector`.

In [4]:
import urllib.request
import json

ES_URL = "http://localhost:9200"
query_vector = [1.0, 0.0, 0.0, 0.0]
knn_body = {
    "knn": {
        "field": "product_vector",
        "query_vector": query_vector,
        "k": 4,
        "num_candidates": 10
    }
}

req = urllib.request.Request(
    f"{ES_URL}/products/_search",
    data=json.dumps(knn_body).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST"
)
with urllib.request.urlopen(req) as resp:
    result = json.load(resp)
for hit in result.get("hits", {}).get("hits", []):
    print(hit["_source"]["name"], "— score:", hit.get("_score"))

URLError: <urlopen error [WinError 10061] Aucune connexion n’a pu être établie car l’ordinateur cible l’a expressément refusée>

## 3. Requête de matching avec `script_score`

On utilise un script Painless pour calculer un score à partir :
- d’un vecteur d’entrée `params.input_vector`,
- du vecteur stocké dans `doc['product_vector']`,
- (optionnel) d’un vecteur de poids `params.weights`.

Une idée simple consiste à :
1. calculer une distance (par ex. euclidienne),
2. transformer cette distance en score, par exemple :  
   \( score = \frac{1}{1 + distance} \).

In [ ]:
query_body = {
    "query": {
        "script_score": {
            "query": {"match_all": {}},
            "script": {
                "source": """
double distance = 0.0;
for (int i = 0; i < params.input_vector.length; ++i) {
  double diff = params.input_vector[i] - doc['product_vector'][i];
  distance += diff * diff;
}
distance = Math.sqrt(distance);
double score = 1.0 / (1.0 + distance);
return score;
                """,
                "params": {
                    "input_vector": [1.0, 0.0, 0.0, 0.0]
                }
            }
        }
    }
}

print(json.dumps(query_body, indent=2, ensure_ascii=False))

## 4. Bonus : pondération construite et statistiques online

Idées de questions :
1. Comment choisir une pondération par dimension (ex. en fonction de la variance) ?
2. Comment standardiser les features \((x - \mu)/\sigma\) ?
3. Comment mettre à jour \(\mu\) et \(\sigma\) au fil de l’eau (algorithme online) ?

Pseudo-code (Welford) pour une seule dimension, à généraliser à tout le vecteur :

```text
count = count + 1
delta = value - mean
mean = mean + delta / count
delta2 = value - mean
M2 = M2 + delta * delta2

if count > 1:
    variance = M2 / (count - 1)
    std = sqrt(variance)
```

Proposez ensuite une stratégie de pondération des dimensions à partir de ces statistiques (par exemple donner plus de poids aux dimensions les plus discriminantes).